In [ ]:
"""
Expected input
--------------
- A CSV file containing features + a binary target column (0/1).
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import Normalizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, recall_score, f1_score, confusion_matrix,
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve
)

# ============================================================
# 1) SETTINGS 
# ============================================================

# Path to the input dataset
CSV_PATH = "Input.csv"

# Name of the target column (binary: 0/1)
CLASS_COL = "Class"

# Outer split seeds = repeated 80/20 splits for robust estimates
OUTER_RANDOM_STATES = [42, 101, 202, 1, 25, 88]

# Feature selection option:
# - None => full feature set 
# - int  => keep only the TOP_K features by train-only Gini importance
TOP_K = None  # example: TOP_K = 25

# Fixed Random Forest parameters
RF_PARAMS = dict(
    n_estimators=400,
    max_depth=6,
    min_samples_split=10,
    min_samples_leaf=8,
    max_features="sqrt",
    class_weight="balanced",
    n_jobs=-1
)

# Optional permutation test (kept off by default; expensive)
RUN_PERMUTATION_TEST = False
N_PERMUTATIONS = 1000
PERM_RANDOM_STATE = 123

# Plot controls
PLOT_PER_SPLIT_CONFUSION = True
SHOW_MEAN_ROC_PR_ONLY = True


# ============================================================
# 2) LOAD DATA (basic validation + split X/y)
# ============================================================

def load_data(csv_path: str, class_col: str):
    """Load CSV and return (X, y, feature_names)."""
    df = pd.read_csv(csv_path)

    if class_col not in df.columns:
        raise ValueError(f"'{class_col}' column not found in CSV.")

    y = df[class_col].astype(int).values
    X_df = df.drop(columns=[class_col]).copy()
    feature_names = X_df.columns.to_list()
    X = X_df.values

    # Enforce 0/1 labels (important for ROC/AUC and consistent reporting)
    if not np.array_equal(np.unique(y), [0, 1]):
        raise ValueError("Class labels must be exactly 0/1.")

    return X, y, feature_names


# ============================================================
# 3) CORE EVALUATION 
# ============================================================

def run_outer_splits(X, y, feature_names):
    """
    Run repeated 80/20 splits:
    - Normalize train, transform test
    - Fit train-only RF for Gini
    - Optionally select TOP_K features based on train-only Gini
    - Fit final RF and evaluate train/test metrics
    - Store curves for mean ROC/PR
    """
    split_metrics = []
    conf_matrices = []
    gini_allfeatures = []
    roc_curves = []
    pr_curves = []

    for split_i, rs in enumerate(OUTER_RANDOM_STATES, start=1):
        # ---------------------------
        # 3.1 Create an outer split
        # ---------------------------
        X_train_raw, X_test_raw, y_train, y_test = train_test_split(
            X, y,
            test_size=0.20,
            random_state=rs,
            stratify=y
        )

        # ------------------------------------------
        # 3.2 Train-only normalization (NO leakage)
        # ------------------------------------------
        normalizer = Normalizer(norm="l2")
        X_train = normalizer.fit_transform(X_train_raw)  # fit ONLY on train
        X_test = normalizer.transform(X_test_raw)        # apply to test

        # ---------------------------------------------------
        # 3.3 Train-only RF to compute Gini importances
        #     (used for (a) plot and (b) TOP_K selection)
        # ---------------------------------------------------
        rf_for_gini = RandomForestClassifier(**RF_PARAMS, random_state=rs)
        rf_for_gini.fit(X_train, y_train)
        importances = rf_for_gini.feature_importances_
        gini_allfeatures.append(importances)

        # -----------------------------------------------
        # 3.4 Choose which features to use for this split
        # -----------------------------------------------
        if TOP_K is None:
            keep_idx = np.arange(X_train.shape[1])  # keep everything
        else:
            # selection is based ONLY on train importances
            keep_idx = np.argsort(importances)[-TOP_K:]
            keep_idx = np.sort(keep_idx)

        X_train_k = X_train[:, keep_idx]
        X_test_k = X_test[:, keep_idx]

        # -----------------------------------
        # 3.5 Final RF (trained on train fold)
        # -----------------------------------
        rf_final = RandomForestClassifier(**RF_PARAMS, random_state=rs)
        rf_final.fit(X_train_k, y_train)

        # --------------------
        # 3.6 Train metrics
        # --------------------
        y_train_pred = rf_final.predict(X_train_k)
        train_acc = accuracy_score(y_train, y_train_pred)
        train_f1 = f1_score(y_train, y_train_pred, pos_label=1)

        # --------------------
        # 3.7 Test metrics
        # --------------------
        y_test_pred = rf_final.predict(X_test_k)
        test_acc = accuracy_score(y_test, y_test_pred)
        test_rec = recall_score(y_test, y_test_pred, pos_label=1)
        test_f1 = f1_score(y_test, y_test_pred, pos_label=1)

        cm = confusion_matrix(y_test, y_test_pred)
        conf_matrices.append(cm)

        # Probabilities for ROC/PR
        y_proba = rf_final.predict_proba(X_test_k)[:, 1]
        roc_auc = roc_auc_score(y_test, y_proba)
        pr_auc = average_precision_score(y_test, y_proba)

        # Store curves 
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        precision, recall, _ = precision_recall_curve(y_test, y_proba)
        roc_curves.append((fpr, tpr))
        pr_curves.append((recall, precision))

        split_metrics.append({
            "Split": split_i,
            "Seed": rs,
            "Num_Features_Used": len(keep_idx),
            "Train_Accuracy": train_acc,
            "Train_F1": train_f1,
            "Test_Accuracy": test_acc,
            "Test_Recall": test_rec,
            "Test_F1": test_f1,
            "Test_ROC_AUC": roc_auc,
            "Test_PR_AUC": pr_auc,
        })

        # Console log
        print(
            f"[Split {split_i} | seed {rs}] "
            f"Train Acc={train_acc:.3f}  Train F1={train_f1:.3f} | "
            f"Test Acc={test_acc:.3f}  Test F1={test_f1:.3f} | "
            f"ROC-AUC={roc_auc:.3f}  PR-AUC={pr_auc:.3f}"
        )

    return split_metrics, conf_matrices, gini_allfeatures, roc_curves, pr_curves


# ============================================================
# 4) REPORTING
# ============================================================

def print_metrics_tables(split_metrics):
    """Print per-split metrics and mean±SD summary."""
    metrics_df = pd.DataFrame(split_metrics)

    print("\n====================")
    print("PER-SPLIT METRICS")
    print("====================")
    print(metrics_df.to_string(index=False))

    summary = metrics_df.drop(columns=["Split", "Seed"]).agg(["mean", "std"])
    print("\n====================")
    print("SUMMARY (MEAN ± SD)")
    print("====================")
    print(summary.to_string())

    return metrics_df, summary


def plot_test_acc_f1_bar(metrics_df):
    """Bar chart showing test accuracy and test F1 per split (manuscript figure)."""
    x = np.arange(len(metrics_df))
    bar_width = 0.35

    plt.figure(figsize=(10, 6))
    plt.bar(
        x - bar_width / 2,
        metrics_df["Test_Accuracy"],
        bar_width,
        label="Test Accuracy",
        color="skyblue",
        edgecolor="black"
    )
    plt.bar(
        x + bar_width / 2,
        metrics_df["Test_F1"],
        bar_width,
        label="Test F1 Score",
        color="salmon",
        edgecolor="black"
    )

    plt.xticks(x, [f"Split {i}" for i in metrics_df["Split"]])
    plt.ylim(0, 1.05)
    plt.xlabel("Data Splits")
    plt.ylabel("Score")
    plt.title("Test Accuracy and F1 Score per Split")
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_confusion_matrices(conf_matrices):
    """Plot a confusion matrix for each outer split."""
    for i, cm in enumerate(conf_matrices, start=1):
        plt.figure(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.title(f"Confusion Matrix – Split {i}")
        plt.tight_layout()
        plt.show()


def plot_log_gini(gini_allfeatures, feature_names, log_gi_cutoff=-11):
    """
    Manuscript-style log-Gini plot:
    - average Gini importance across splits (train-only importances)
    - log transform for visualization
    - apply cutoff so tiny values don't flatten the plot
    """
    mean_gini = np.mean(np.vstack(gini_allfeatures), axis=0)

    gini_df = (
        pd.DataFrame({"Metabolite": feature_names, "GI": mean_gini})
        .sort_values("GI")
        .reset_index(drop=True)
    )

    gini_df["logGI"] = np.log(gini_df["GI"] + 1e-20)

    gini_df_plot = gini_df[gini_df["logGI"] >= log_gi_cutoff].copy()

    print(
        f"\nGini plot: showing {len(gini_df_plot)} features out of {len(gini_df)} "
        f"(logGI >= {log_gi_cutoff})"
    )

    plt.figure(figsize=(12, 6))
    plt.plot(
        range(len(gini_df_plot)),
        gini_df_plot["logGI"].values,
        marker="o",
        linestyle="-",
        color="blue"
    )
    plt.scatter(
        range(len(gini_df_plot)),
        gini_df_plot["logGI"].values,
        s=20,
        color="blue"
    )

    plt.xlabel("Metabolites / Features")  
    plt.ylabel("Log Gini Index")
    plt.title("Mean Gini Index Across Metabolites/Features (Train-only)")
    plt.xticks([])
    plt.grid(axis="y", linestyle="--", alpha=0.7)
    plt.tight_layout()
    plt.show()

    # Print top 25 (unfiltered)
    print("\nTop 25 features by mean Gini (train-only):")
    print(gini_df.sort_values("GI", ascending=False).head(25).to_string(index=False))

    return gini_df


def plot_mean_roc(roc_curves):
    """Plot a single mean ROC curve across splits."""
    mean_fpr = np.linspace(0, 1, 300)
    mean_tpr = np.mean(
        [np.interp(mean_fpr, fpr, tpr) for fpr, tpr in roc_curves],
        axis=0
    )

    plt.figure(figsize=(7, 6))
    plt.plot(mean_fpr, mean_tpr, label="Mean ROC")
    plt.plot([0, 1], [0, 1], linestyle="--", label="Chance")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("Mean ROC Curve Across Outer Splits")
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_mean_pr(pr_curves):
    """Plot a single mean precision–recall curve across splits."""
    mean_recall = np.linspace(0, 1, 300)
    mean_precision = np.mean(
        [np.interp(mean_recall, r[::-1], p[::-1]) for r, p in pr_curves],
        axis=0
    )

    plt.figure(figsize=(7, 6))
    plt.plot(mean_recall, mean_precision, label="Mean PR")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Mean Precision–Recall Curve Across Outer Splits")
    plt.legend()
    plt.tight_layout()
    plt.show()
